<a href="https://colab.research.google.com/github/GraceMolina/SQL-Problem-Solving/blob/main/ReadingAndFilteringDataFromAnOnlineStore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# mount google drive to read the dataset
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# read the dataset
import pandas as pd

file_path = '/content/drive/MyDrive/SQL Problem Solving/Module 1/Dataset.xlsx'

data = pd.ExcelFile(file_path)
print(data.sheet_names)

['items', 'users', 'orders', 'line_items', 'events', 'item_field_history', 'dates']


In [ ]:
# read sheets
sheets = pd.read_excel(file_path, sheet_name = None)

for name, df in sheets.items():
  print(name, df.shape)
  display(df.head())

In [ ]:
# read the sheets into a SQL database
import sqlite3

conn = sqlite3.connect(':memory:')

for name, df in sheets.items():
  table = name.strip().replace(' ', '_')
  df.to_sql(table, conn, index = False, if_exists = 'replace')

# confirm the tables exist
pd.read_sql('SELECT name FROM sqlite_master WHERE type = "table"', conn)

In [ ]:
# select all the items in one category of items
query1 = """
SELECT *
FROM items
WHERE category = 'Connector'
"""

result1 = pd.read_sql(query1, conn)
result1

In [ ]:
# count the number of orders placed by day
query2 = """
SELECT DATE(created_timestamp) as order_date,
       COUNT(*)                as num_orders
FROM orders
GROUP BY order_date
ORDER BY order_date
"""

result2 = pd.read_sql(query2, conn)
result2

In [ ]:
# set price buckets
query3 = """
SELECT list_price,
  CASE
    WHEN list_price < 25 THEN '0-24.99'
    WHEN list_price < 50 THEN '25-49.99'
    WHEN list_price < 75 THEN '50-74.99'
    WHEN list_price < 100 THEN '75-99.99'
    WHEN list_price < 125 THEN '100-124.99'
    WHEN list_price < 150 THEN '125-149.99'
    WHEN list_price < 175 THEN '150-174.99'
    ELSE '175+'
  END AS price_bucket
FROM items
"""

result3 = pd.read_sql(query3, conn)
result3

In [ ]:
# count the number of times an ordered item has been ordered,
# including the name of the item
query4 = """
SELECT item_id,
       COUNT(*) AS times_ordered,
       items.display_name
FROM line_items
JOIN items
ON line_items.item_id = items.id
GROUP BY item_id
"""

result4 = pd.read_sql(query4, conn)
result4

In [ ]:
# count the number of times an item has been ordered,
# including the name of the item
query5 = """
SELECT items.id,
       items.display_name,
       COUNT(line_items.item_id) AS times_ordered
FROM items
LEFT JOIN line_items
ON items.id = line_items.item_id
GROUP BY line_items.item_id
"""

result5 = pd.read_sql(query5, conn)
result5

In [ ]:
# for each user, calculate the average number of items per order
query6 = """
WITH average_items_per_order AS (SELECT orders.user_id,
       AVG(items_per_order.items_ordered) AS avg_items_per_order
FROM orders
LEFT JOIN (SELECT order_id, COUNT(*) AS items_ordered
           FROM line_items
           GROUP BY order_id) AS items_per_order
ON orders.id = items_per_order.order_id
GROUP BY orders.user_id)
SELECT users.id,
       average_items_per_order.avg_items_per_order
FROM users
LEFT JOIN average_items_per_order
ON users.id = average_items_per_order.user_id
"""

result6 = pd.read_sql(query6, conn)
result6

In [ ]:
# return users who have purchased at least one item
# approach 1
query8 = """
SELECT DISTINCT users.id AS user_id
FROM users
INNER JOIN orders
ON users.id = orders.user_id
"""

result8 = pd.read_sql(query8, conn)
result8

In [ ]:
# approach 2
query9 = """
SELECT DISTINCT users.id AS user_id
FROM users
JOIN orders
ON users.id = orders.user_id
"""

result9 = pd.read_sql(query9, conn)
result9

In [ ]:
# calcualte the average number of items per order
query10 = """
SELECT AVG(items_ordered) AS average_number_of_items
FROM (SELECT order_id,
             COUNT(item_id) AS items_ordered
     FROM line_items
     GROUP BY order_id)
"""

result10 = pd.read_sql(query10, conn)
result10